# Exam-Scheduling GPU Measurement — Colab T4

**Self-validating measurement harness.** Each section has a PASS/FAIL check; the notebook halts early if a prerequisite fails so you don't waste runtime.

## What this notebook does

1. **Verify GPU + nvcc** available at runtime.
2. **Clone repo + build** the CUDA library and main binary with `HAVE_CUDA=1`.
3. **Run `make bench HAVE_CUDA=1`** — kernel vs CPU-twin validators (move-delta, placement, full-eval, per-student semantic parity). All must PASS.
4. **Confirm `gpu=on`** at runtime for every `*_cuda` variant.
5. **Benchmark sweep:** all 7 (parent, cuda) pairs × 5 ITC instances × 3 seeds. Records runtime, soft, hard, feasibility.
6. **Quality-parity assertion:** same seed → same soft (bit-exact). Fails loudly if any variant drifts.
7. **Summary tables + plots:** GPU/CPU speedup heatmap, quality-parity grid, and per-variant runtime bars.
8. **Save artifacts** to Drive (optional) + download link.

## 0. Edit these before running

In [ ]:
REPO_URL = 'https://github.com/Dialovos/exam-scheduling-algos-analyze'
BRANCH   = 'master'

INSTANCES = [1, 4, 5, 7, 8]   # ITC 2007 set numbers
SEEDS     = [42, 43, 44]       # 3 seeds for averaging
SAVE_TO_DRIVE = False          # True mounts Drive and saves CSVs there
DRIVE_OUT_DIR = '/content/drive/MyDrive/exam-scheduling-results'

# Iter budgets per algo — keep per-variant sensible; total run ~= 30-60 min on T4.
VARIANT_PAIRS = [
    # (parent,         cuda,               iter_flag,       iters)
    ('tabu_cached',    'tabu_cuda',        'tabu-iters',    1000),
    ('sa_cached',      'sa_parallel_cuda', 'sa-iters',      5000),
    ('alns_thompson',  'alns_cuda',        'alns-iters',    500),
    ('ga',             'ga_cuda',          'ga-iters',      100),
    ('abc',            'abc_cuda',         'abc-iters',     500),
    ('hho',            'hho_cuda',         'hho-iters',     100),
    ('woa',            'woa_cuda',         'woa-iters',     300),
]

## 1. GPU + nvcc environment checks (must PASS before continuing)

In [ ]:
import subprocess, sys
def run(cmd, timeout=180, check=True, capture=True):
    r = subprocess.run(cmd, shell=True, timeout=timeout,
                        capture_output=capture, text=True)
    if check and r.returncode != 0:
        print(r.stdout); print(r.stderr, file=sys.stderr)
        raise RuntimeError(f'Command failed: {cmd}')
    return r

print('=== GPU check ===')
r = run('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader', check=False)
if r.returncode != 0:
    raise RuntimeError('No GPU detected — switch runtime to T4/A100.')
print(r.stdout.strip())

print('\n=== nvcc check ===')
r = run('nvcc --version | tail -2', check=False)
if r.returncode != 0 or 'release' not in r.stdout:
    raise RuntimeError('nvcc not available — Colab should have it; try a different runtime.')
print(r.stdout.strip())

print('\n=== libcudart probe ===')
r = run('find /usr -name "libcudart.so*" | head -3', check=False)
print(r.stdout.strip() or '(none found in /usr — build will still try /usr/local/cuda/lib64)')

print('\n✓ Environment checks passed.')

## 2. Clone + build (HAVE_CUDA=1)

In [ ]:
import os
REPO_DIR = '/content/exam-scheduling'

if not os.path.isdir(REPO_DIR):
    run(f'git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}')
else:
    run(f'cd {REPO_DIR} && git pull', check=False)
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
run('ls -la instances | head -15')

In [ ]:
print('=== make cuda-build ===')
r = run('make cuda-build', timeout=300)
print(r.stdout[-400:])
assert os.path.exists('cpp/build/libdelta_cuda.so'), 'libdelta_cuda.so was not built'
print('✓ libdelta_cuda.so built')

print('\n=== make all HAVE_CUDA=1 ===')
r = run('make all HAVE_CUDA=1', timeout=600)
print(r.stdout[-400:])
assert os.path.exists('cpp/build/exam_solver'), 'exam_solver was not built'
print('✓ exam_solver built with HAVE_CUDA=1')

## 3. Runtime sanity: confirm `gpu=on` for every `*_cuda` variant

In [ ]:
import re
def gpu_status(algo, flag, iters=10):
    r = run(f'./cpp/build/exam_solver instances/exam_comp_set4.exam '
            f'--algo {algo} --seed 42 --{flag} {iters} -v 2>&1 | grep -E "gpu="',
            check=False)
    m = re.search(r'gpu=(on|off)', r.stdout)
    return m.group(1) if m else 'unknown'

checks = [
    ('tabu_cuda',        'tabu-iters'),
    ('sa_parallel_cuda', 'sa-iters'),
    ('alns_cuda',        'alns-iters'),
    ('ga_cuda',          'ga-iters'),
    ('abc_cuda',         'abc-iters'),
    ('hho_cuda',         'hho-iters'),
    ('woa_cuda',         'woa-iters'),
]
all_on = True
for algo, flag in checks:
    status = gpu_status(algo, flag)
    marker = '✓' if status == 'on' else '✗'
    print(f'  {marker} {algo:22s} gpu={status}')
    if status != 'on': all_on = False

assert all_on, 'Some variants did not activate GPU — check libdelta_cuda linkage'
print('\n✓ All variants report gpu=on')

## 4. Kernel-vs-CPU-twin validators (`make bench HAVE_CUDA=1`)

All four kernels must match their CPU twins bit-exact. Any FAIL halts the notebook.

In [ ]:
print('Building bench with HAVE_CUDA=1...')
run('rm -f cpp/build/bench_eval && make bench HAVE_CUDA=1', timeout=600)

print('\n=== Running validators ===')
r = run('./cpp/build/bench_eval instances/exam_comp_set4.exam 100 2>&1 '
         '| grep -E "validator|VERDICT|mismatches|PASS|FAIL"', timeout=120)
print(r.stdout)

fail_count = r.stdout.count('FAIL')
assert fail_count == 0, f'{fail_count} validator(s) FAILED — fix kernel before proceeding'
pass_count = r.stdout.count('PASS')
print(f'\n✓ All {pass_count} validators PASSED')

## 5. Benchmark sweep

All (parent, cuda) pairs × instances × seeds. Records full JSON output per run.

In [ ]:
import json, time, csv
import pandas as pd

# All artifacts land under results/gpu_measurement_colab/ to mirror the
# existing batch_018_colab / batch_019_validation convention in this repo.
OUT_DIR = 'results/gpu_measurement_colab'
os.makedirs(OUT_DIR, exist_ok=True)

def solve(instance, algo, seed, flag, iters, timeout_s=1200):
    cmd = (f'./cpp/build/exam_solver instances/exam_comp_set{instance}.exam '
           f'--algo {algo} --seed {seed} --{flag} {iters}')
    t0 = time.time()
    try:
        r = subprocess.run(cmd, shell=True, timeout=timeout_s,
                            capture_output=True, text=True)
        if r.returncode != 0:
            return {'error': f'exit {r.returncode}: {r.stderr[:200]}'}
        data = json.loads(r.stdout)
        out = data[0] if isinstance(data, list) and data else data
        return {
            'runtime_s': out.get('runtime', time.time() - t0),
            'soft': int(out.get('soft_penalty', -1)),
            'hard': int(out.get('hard_violations', -1)),
            'feasible': bool(out.get('feasible', False)),
        }
    except subprocess.TimeoutExpired:
        return {'error': 'timeout'}
    except Exception as e:
        return {'error': str(e)}

# Per-run CSV flush — protects against Colab disconnects mid-sweep.
RAW_CSV = f'{OUT_DIR}/gpu_sweep_raw.csv'
RAW_COLS = ['set', 'algo', 'seed', 'iters', 'pair_parent',
            'runtime_s', 'soft', 'hard', 'feasible', 'error']

with open(RAW_CSV, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=RAW_COLS).writeheader()

rows = []
total_runs = len(INSTANCES) * len(VARIANT_PAIRS) * len(SEEDS) * 2
run_idx = 0
t_bench_start = time.time()

for inst in INSTANCES:
    for cpu_algo, gpu_algo, flag, iters in VARIANT_PAIRS:
        for seed in SEEDS:
            for variant in (cpu_algo, gpu_algo):
                run_idx += 1
                res = solve(inst, variant, seed, flag, iters)
                row = {'set': f'set{inst}', 'algo': variant, 'seed': seed,
                       'iters': iters, 'pair_parent': cpu_algo,
                       'runtime_s': res.get('runtime_s'),
                       'soft': res.get('soft'), 'hard': res.get('hard'),
                       'feasible': res.get('feasible'),
                       'error': res.get('error', '')}
                rows.append(row)
                # Append this row to CSV immediately (safe against disconnect)
                with open(RAW_CSV, 'a', newline='') as f:
                    csv.DictWriter(f, fieldnames=RAW_COLS).writerow(row)

                err = res.get('error', '')
                if err:
                    print(f'[{run_idx}/{total_runs}] set{inst} {variant} seed={seed} ERROR: {err}')
                else:
                    print(f'[{run_idx}/{total_runs}] set{inst} {variant:22s} seed={seed} '
                          f'rt={res["runtime_s"]:.2f} soft={res["soft"]} '
                          f'feas={res["feasible"]}')

df = pd.DataFrame(rows)
print(f'\n✓ Sweep done in {time.time() - t_bench_start:.0f}s — {len(df)} rows written incrementally to {RAW_CSV}')
df.head(10)

## 6. Quality-parity assertion (CPU vs GPU must match bit-exact per seed)

Phase 2 (per-student conflict semantics) guarantees same-seed parity. Any drift indicates a regression.

In [ ]:
parity = []
for (inst, seed, parent), grp in df.groupby(['set', 'seed', 'pair_parent']):
    gpu_algo = next(g for (p, g, _, _) in VARIANT_PAIRS if p == parent)
    cpu_rows = grp[grp['algo'] == parent]
    gpu_rows = grp[grp['algo'] == gpu_algo]
    if cpu_rows.empty or gpu_rows.empty: continue
    if 'error' in cpu_rows.columns:
        if cpu_rows['error'].notna().any() or gpu_rows['error'].notna().any():
            continue
    cpu_soft = cpu_rows['soft'].iloc[0]
    gpu_soft = gpu_rows['soft'].iloc[0]
    parity.append({
        'set': inst, 'seed': seed, 'pair': parent,
        'cpu_soft': cpu_soft, 'gpu_soft': gpu_soft,
        'diff': gpu_soft - cpu_soft,
        'match': cpu_soft == gpu_soft,
    })

parity_df = pd.DataFrame(parity)
mismatches = parity_df[~parity_df['match']]
print(f'Total pairs: {len(parity_df)}  |  Bit-exact matches: {parity_df["match"].sum()}  |  Mismatches: {len(mismatches)}')

if not mismatches.empty:
    print('\n⚠ MISMATCHES:')
    print(mismatches.to_string(index=False))
    # Documented-acceptable deviations (see PERF_ROADMAP.md)
    ACCEPTABLE_PARENTS = {
        'sa_cached',      # sa_parallel_cuda uses adj-based SA (experimental Phase 4)
        'ga',             # ga_cuda init-pop reordering can diverge on some seeds (minor)
    }
    hard_fail = mismatches[~mismatches['pair'].isin(ACCEPTABLE_PARENTS)]
    if hard_fail.empty:
        print('\n✓ All mismatches are in the documented-acceptable list — no regression')
    else:
        print(f'\n✗ UNEXPECTED divergence on {len(hard_fail)} production pairs:')
        print(hard_fail.to_string(index=False))
        raise AssertionError('quality parity regression — investigate')
else:
    print('\n✓ All pairs match bit-exact per seed')

## 7. Summary — GPU/CPU speedup per (set, pair)

In [ ]:
summary_rows = []
for inst in INSTANCES:
    for cpu_algo, gpu_algo, _, _ in VARIANT_PAIRS:
        cpu = df[(df['set'] == f'set{inst}') & (df['algo'] == cpu_algo)]
        gpu = df[(df['set'] == f'set{inst}') & (df['algo'] == gpu_algo)]
        if cpu.empty or gpu.empty: continue
        cpu_rt = cpu['runtime_s'].astype(float).mean()
        gpu_rt = gpu['runtime_s'].astype(float).mean()
        cpu_soft = cpu['soft'].astype(float).mean()
        gpu_soft = gpu['soft'].astype(float).mean()
        summary_rows.append({
            'set': f'set{inst}', 'pair': f'{cpu_algo} → {gpu_algo}',
            'cpu_rt_s': round(cpu_rt, 2), 'gpu_rt_s': round(gpu_rt, 2),
            'speedup_gpu/cpu': round(cpu_rt / gpu_rt, 2) if gpu_rt > 0 else None,
            'cpu_mean_soft': round(cpu_soft, 1), 'gpu_mean_soft': round(gpu_soft, 1),
            'soft_diff': round(gpu_soft - cpu_soft, 1),
            'gpu_wins': cpu_rt / gpu_rt > 1.05 if gpu_rt > 0 else False,
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f'{OUT_DIR}/gpu_sweep_summary.csv', index=False)
summary_df

## 8. Plots — speedup heatmap + runtime bars

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Heatmap: speedup per (set, pair). >1 means GPU faster.
pivot = summary_df.pivot(index='pair', columns='set', values='speedup_gpu/cpu')
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0, vmax=2, aspect='auto')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                     color='black' if 0.6 < v < 1.4 else 'white', fontsize=9)
ax.set_title('GPU speedup per (algo pair, instance)  —  >1 means GPU faster')
plt.colorbar(im, ax=ax, label='CPU_rt / GPU_rt')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/gpu_speedup_heatmap.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
# Bar chart: mean runtime per variant across all instances
fig, ax = plt.subplots(figsize=(11, 5))
cpu_means = [df[df['algo'] == p]['runtime_s'].astype(float).mean() for p, _, _, _ in VARIANT_PAIRS]
gpu_means = [df[df['algo'] == g]['runtime_s'].astype(float).mean() for _, g, _, _ in VARIANT_PAIRS]
labels = [f'{p}\n→{g}' for p, g, _, _ in VARIANT_PAIRS]
x = np.arange(len(labels)); w = 0.35
ax.bar(x - w/2, cpu_means, w, label='CPU', color='#3b82f6')
ax.bar(x + w/2, gpu_means, w, label='GPU', color='#f97316')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Mean runtime (s)'); ax.set_title('Mean runtime across all tested instances')
ax.legend(); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/gpu_runtime_bars.png', dpi=110, bbox_inches='tight')
plt.show()

## 9. SA-parallel throughput measurement (separate — not an apples-to-apples CPU comparison)

`sa_parallel_cuda` launches N_seeds × n_iters in one kernel. Measuring raw SA iter/sec is the paper-grade number.

In [ ]:
import time
sa_throughput_rows = []
for inst in INSTANCES[:3]:   # subset — each run is fast
    for sa_iters in [2000, 5000, 10000]:
        n_seeds = 64  # hard-coded in sa_parallel_cuda.h
        t0 = time.time()
        r = solve(inst, 'sa_parallel_cuda', 42, 'sa-iters', sa_iters, timeout_s=600)
        rt = r.get('runtime_s', time.time() - t0)
        total_iters = n_seeds * sa_iters
        throughput = total_iters / rt if rt > 0 else 0
        sa_throughput_rows.append({
            'set': f'set{inst}', 'n_seeds': n_seeds, 'iters_per_seed': sa_iters,
            'total_iters': total_iters, 'runtime_s': round(rt, 2),
            'iter_per_sec': int(throughput),
            'feasible': r.get('feasible', False), 'soft': r.get('soft', -1),
        })
        print(f'set{inst} × {n_seeds} seeds × {sa_iters} iters = {total_iters} in {rt:.2f}s → {int(throughput):,} iter/sec  feas={r.get("feasible")}')

sa_df = pd.DataFrame(sa_throughput_rows)
sa_df.to_csv(f'{OUT_DIR}/sa_parallel_throughput.csv', index=False)
sa_df

## 10. Save / sync artifacts

Options:

**A. Download the folder directly** — in the Colab file browser (left sidebar), right-click `exam-scheduling/results/gpu_measurement_colab/` → *Download*. You get a zip; extract it into your local repo's `results/` folder.

**B. Save to Drive** — set `SAVE_TO_DRIVE = True` at the top. Mounts your Drive and copies the folder to `DRIVE_OUT_DIR`.

**C. Commit from Colab** — if you set up a GitHub token in Colab secrets, you can `git push` straight back to your fork. Instructions in the cell below the next one.

In [ ]:
# Generate a README in the results subfolder describing each artifact
from datetime import datetime, timezone

gpu_name = run('nvidia-smi --query-gpu=name --format=csv,noheader').stdout.strip()
cuda_ver = run('nvcc --version | grep -oE "V[0-9.]+" | head -1').stdout.strip()
timestamp = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')

readme = f'''# gpu_measurement_colab — {timestamp}

Benchmark sweep of GPU vs CPU variants for the exam-scheduling solver.

**Environment**
- GPU: `{gpu_name}`
- CUDA: `{cuda_ver}`
- Generated by: `notebooks/gpu_measurement_colab.ipynb`

**Config**
- Instances: `{INSTANCES}`
- Seeds: `{SEEDS}`
- Variant pairs: `{[(p, g) for (p, g, _, _) in VARIANT_PAIRS]}`

**Artifacts**

| File | Content |
|---|---|
| `gpu_sweep_raw.csv` | One row per run: `(set, algo, seed, iters, pair_parent, runtime_s, soft, hard, feasible, error)`. Flushed incrementally — survives mid-sweep Colab disconnects. |
| `gpu_sweep_summary.csv` | Aggregated speedup per `(set, pair)`: `cpu_rt_s`, `gpu_rt_s`, `speedup_gpu/cpu`, quality diff, `gpu_wins` boolean. |
| `sa_parallel_throughput.csv` | `sa_parallel_cuda` raw throughput: iter/sec at several `(n_seeds, iters_per_seed)` sizes. |
| `gpu_speedup_heatmap.png` | Speedup heatmap (green = GPU faster). |
| `gpu_runtime_bars.png` | CPU vs GPU mean runtime bars across all instances. |

**Parity status**

All `*_cuda` variants produce bit-exact fitness with their CPU parents on same seed, *except*:
- `sa_parallel_cuda` — uses adj-based conflict semantics (experimental, Phase 4). Will differ from `sa_cached`.
- `ga_cuda` — minor init-pop reordering can shift trajectory on some (seed, instance) combos.

See `docs/PERF_ROADMAP.md` for the full kernel-vs-twin validation methodology.
'''

with open(f'{OUT_DIR}/README.md', 'w') as f:
    f.write(readme)
print(f'✓ Wrote {OUT_DIR}/README.md')
run(f'ls -la {OUT_DIR}')

In [ ]:
# Option B: Drive save
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dest = f'{DRIVE_OUT_DIR}/gpu_measurement_colab'
    os.makedirs(drive_dest, exist_ok=True)
    run(f'cp -r {OUT_DIR}/* {drive_dest}/', check=False)
    print(f'✓ Saved to {drive_dest}')
else:
    print('SAVE_TO_DRIVE=False — artifacts stay in the Colab runtime.')
    print(f'   Location: /content/exam-scheduling/{OUT_DIR}/')
    print(f'   Download via the left-sidebar file browser, or set SAVE_TO_DRIVE=True.')

print('\n--- Local merge instructions ---')
print('After downloading (or syncing from Drive), put the folder into your local repo:')
print()
print('    cd /path/to/local/exam-scheduling')
print('    # if downloaded as zip:')
print('    unzip ~/Downloads/gpu_measurement_colab.zip -d results/')
print('    # or if copied from Drive:')
print('    cp -r ~/GoogleDrive/exam-scheduling-results/gpu_measurement_colab results/')
print()
print('    git add results/gpu_measurement_colab')
print('    git commit -m "results: GPU measurement sweep (Colab T4)"')
print('    git push')

## 11. Final report

Copy-paste-friendly summary with all the numbers that matter.

In [ ]:
print('=' * 70)
print(' GPU MEASUREMENT REPORT')
print('=' * 70)

print(f'\nGPU: {gpu_name}   CUDA: {cuda_ver}')
print(f'Instances: {INSTANCES}   Seeds: {SEEDS}')
print(f'Total runs: {len(df)}')
print(f'Artifacts dir: {OUT_DIR}/')

print('\n--- Quality parity ---')
print(f'  Bit-exact per-seed matches: {parity_df["match"].sum()}/{len(parity_df)}')

print('\n--- GPU wins (speedup > 1.05×) ---')
wins = summary_df[summary_df['gpu_wins']]
if wins.empty:
    print('  (none — tighter GPU integration needed)')
else:
    for _, r in wins.iterrows():
        print(f'  {r["pair"]:40s} set={r["set"]:5s} {r["speedup_gpu/cpu"]:.2f}× faster')

print('\n--- SA-parallel throughput ---')
max_tp = sa_df['iter_per_sec'].max()
print(f'  Peak: {int(max_tp):,} SA iter/sec on this GPU')

print('\n--- Artifacts in', OUT_DIR, '---')
run(f'ls -la {OUT_DIR}')
print('=' * 70)